# 01 - Collect the four sources
Run with `SAMPLE = True` first. Record what answered, how many rows, what failed.

Day-1 checks before the full pull:
1. NVD change history: do `CVSS V3.1` and `CWE` details carry vectors, source and dates as `data.parse_history_events` assumes?
2. Wayback: do KEV snapshots cover 2022-2024? (cisagov/kev-data history starts on 2025-01-27.)

In [ ]:
# Colab setup: repository at /content/kev-text (git clone or unzip), then run all.
import os, sys
REPO = "/content/kev-text"
if os.path.exists(REPO):
    os.chdir(REPO)
sys.path.insert(0, os.getcwd())
# !pip install -q -r requirements.txt          # once per Colab session
import numpy as np, pandas as pd
from src import config, io_utils, labels, evaluate
SYN = False   # True = dry run on src.synthetic; everything goes to outputs_synthetic/

In [ ]:
from src import data
io_utils.env_lock()
SAMPLE = True
YEARS = range(2021, 2026)

## KEV: the current catalogue is the label snapshot

In [ ]:
kev_file, kev_snapshot_date = data.fetch_kev()
kev = data.parse_kev(io_utils.read_json(kev_file))
print(len(kev), kev.attrs, kev_snapshot_date)

## Day-1 check A: Wayback coverage of KEV snapshots

In [ ]:
cdx = data.kev_wayback_snapshots("2021-11-01")
print(len(cdx), cdx["date"].min(), cdx["date"].max())
cdx.groupby(cdx["date"].dt.to_period("Q")).size()     # S6 needs every quarter of 2022-2024

## cvelistV5 (current CNA records)

In [ ]:
repo = data.fetch_cvelist()
cves = data.load_cvelist(repo, YEARS)
if SAMPLE:
    cves = cves.sample(10, random_state=0)
cves["state"].value_counts(), cves["lang_ok"].mean()

## Day-1 check B: NVD change history on three days

In [ ]:
files = data.fetch_history("2024-03-01", "2024-03-03")
ev = data.parse_history_files(files)
print(ev["type"].value_counts().head(15))
ev[ev["type"].fillna("").str.startswith("CVSS")].head(10)

## Full pulls (resumable: re-run after a Colab disconnect)

In [ ]:
END = str(pd.Timestamp.now().date())
hist_files = files if SAMPLE else data.fetch_history("2021-11-01", END)
events = data.parse_history_files(hist_files)
nvd_files = [] if SAMPLE else data.fetch_nvd_cves("2022-01-01", "2025-06-30")
nvd_now = data.parse_nvd_files(nvd_files) if nvd_files else pd.DataFrame(
    columns=["cve_id", "nvd_published", "nvd_status_now", "desc_current", "cvss_present_now"])

## EPSS at t_pred, merge, save

In [ ]:
cves["published_date"] = cves["published_date"].dt.normalize()
cves["t_pred"] = cves["published_date"] + pd.Timedelta(days=config.T_PRED_OFFSET_DAYS)
epss = data.epss_at_tpred(cves[["cve_id", "t_pred"]])
df = data.build_dataset(cves, events, nvd_now, kev, epss)
df["kev_snapshot_date"] = kev_snapshot_date
print(len(df), "restored:", df["desc_original"].notna().mean(), "epss:", df["epss_at_pred"].notna().mean())
if not SAMPLE:
    df.to_parquet(config.DATASET)
    io_utils.manifest_record(config.DATASET, "built by 01_collect")